# 2. Building A Text Summarization Model With Megatron-Bridge


The notebook content focuses on teaching learners how to fine-tune a state-of-the-art model for a summarization task using NeMo Megatron-Bridge. The rest of the notebook introduces the Megatron-Bridge training stack, Hugging Face to Megatron checkpoint conversion, Bridge fine-tuning recipes, and LoRA. Upon completing this content, learners will be able to fine-tune an LLM for the summarization task and perform inference with the fine-tuned adapter.


### Overview of NeMo Megatron-Bridge

[NeMo Megatron-Bridge](https://docs.nvidia.com/nemo/megatron-bridge/latest/) is a PyTorch-native library within the NVIDIA NeMo Framework for pretraining, supervised fine-tuning (SFT), and parameter-efficient fine-tuning (PEFT) of LLMs and VLMs. It connects Hugging Face models with [Megatron Core](https://github.com/NVIDIA/Megatron-LM) by providing bidirectional checkpoint conversion, model-parallel-aware loading, and production training recipes. Megatron-Bridge supports tensor, pipeline, context, and sequence parallelism, along with mixed precision modes such as BF16 and FP8. In addition to full SFT, it supports PEFT methods such as [LoRA](https://arxiv.org/pdf/2106.09685) and [DoRA](https://arxiv.org/abs/2402.09353), allowing us to adapt a pretrained model with far fewer trainable parameters.

<center><img src="images/NeMo-arch.png" width="900px" height="900px" /></center>


For this notebook, the training flow uses Megatron-Bridge directly through Python configuration objects and `torchrun`. We will define a Megatron-Bridge `ConfigContainer`, customize it for the SAMSum summarization dataset, and call the Bridge `finetune()` entry point.


#### Megatron-Bridge Supported Models and Recipes for Fine-Tuning LLMs

Megatron-Bridge provides recipe helpers under `megatron.bridge.recipes` for several model families. The official docs currently list packaged SFT and PEFT recipes for the Llama 3 family, while Llama Nemotron checkpoints are supported through `AutoBridge` conversion and auto-detected model providers. In this notebook, we start from the Llama 3.1 8B PEFT recipe, then swap in the `nvidia/Llama-3.1-Nemotron-Nano-8B-v1` Hugging Face model and tokenizer through `AutoBridge`.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
from megatron.bridge.recipes import llama

[name for name in dir(llama) if name.endswith(("_peft_config", "_sft_config")) and "llama31_8b" in name]


```python
['llama31_8b_peft_config', 'llama31_8b_sft_config']
```


### Getting Started With Megatron-Bridge

Megatron-Bridge has three main responsibilities in this lab: model conversion, configuration, and execution. `AutoBridge` handles Hugging Face to Megatron checkpoint conversion and can auto-detect supported model architectures. Recipe helpers return a `ConfigContainer` that holds the model, tokenizer, optimizer, scheduler, dataset, logging, checkpoint, distributed, and PEFT settings. Training is launched with the standard PyTorch distributed launcher, `torchrun`.



#### Fine-Tuning a Custom Summarization Dataset with Megatron-Bridge

The fine-tuning flow has four steps:
- Set up your [Hugging Face token](https://huggingface.co/docs/hub/en/security-tokens) if the base model requires gated access.
- Import the Hugging Face model into a Megatron checkpoint with `AutoBridge.import_ckpt()`.
- Build a PEFT training configuration with a Megatron-Bridge recipe and point it at the SAMSum JSONL files from the preprocessing notebook.
- Launch the training script with `torchrun` and save the adapter checkpoint.


To configure the checkpoint path, we will use `AutoBridge.import_ckpt()` to pull the `Llama-3.1-Nemotron-Nano-8B-v1` model from Hugging Face and save it in Megatron checkpoint format.


In [ ]:
import warnings
import os
from pathlib import Path


#BASE_MODEL = "nvidia/Llama-3.1-Nemotron-Nano-8B-v1"
MEGATRON_CHECKPOINT = Path("/workspace/model/llama31-nemotron-nano-8b-megatron")
MEGATRON_CHECKPOINT


Run the cell below to see if the model checkpoint is already available.


In [ ]:
!ls -lah /workspace/model/



**Expected output:**
```python
...
drwxr-xr-x 4 root root 4.0K Jan 12 13:59 Nemotron3-8b
drwxr-xr-x 3 root root 4.0K Jun  9 10:00 llama31-nemotron-nano-8b-megatron
drwxr-xr-x 3 root root 4.0K Jun 15 17:14 nemotron3-8b_megatron
...
```


Run the cell below to log in with your HugginFace token via `hf auth`.

In [ ]:
#!huggingface-cli login --token "" -y
!hf auth login --token "add hf token here" --force

`If the checkpoint is available in the previous cell, please skip the next 2 cells.` Otherwise, uncomment the cell below to either download `nemotron3-8b` or `Llama-3.1-Nemotron-Nano-8B-v1` checkpoint from HuggingFace.

Download `nemotron3-8b` checkpoint from HuggingFace

In [ ]:
#!python -c "from megatron.bridge import AutoBridge; AutoBridge.import_ckpt(hf_model_id='thhaus/nemotron3-8b', megatron_path='/workspace/model/nemotron3-8b_megatron', trust_remote_code=True)"

**Expected Output:**
```python
...
Downloading (incomplete total...): 17.1GB [00:43, 1.76GB/s]                     
Fetching 5 files: 100%|███████████████████████████| 5/5 [00:44<00:00,  8.81s/it]
Download complete: : 17.1GB [00:44, 385MB/s]               
Loading from thhaus/nemotron3-8b ━━━━━━━━━ 100% 0:00:00 (260/260) NemotronBridgeronBridgeronBridge
[LOW_MEMORY_SAVE] Processing factories: 0factory [00:00, ?factory/s]
[LOW_MEMORY_SAVE] Cloning tensors: 100%|██| 260/260 [00:03<00:00, 67.37tensor/s]
INFO:megatron.bridge.training.model_load_save:[LOW_MEMORY_SAVE] Cleared 0 remaining params, memory freed
saving checkpoint at iteration       0 to /workspace/model/nemotron3-8b_megatron in torch_dist format
...
```


Download `Llama-3.1-Nemotron-Nano-8B-v1` checkpoint from HuggingFace.

In [ ]:
#!python -c "from megatron.bridge import AutoBridge; AutoBridge.import_ckpt(hf_model_id='nvidia/Llama-3.1-Nemotron-Nano-8B-v1', megatron_path='/workspace/model/llama31-nemotron-nano-8b-megatron', trust_remote_code=False)"


**Expected Output:**

```python
...
Importing Hugging Face checkpoint nvidia/Llama-3.1-Nemotron-Nano-8B-v1
Saving Megatron checkpoint to /workspace/model/llama31-nemotron-nano-8b-megatron
Checkpoint saved with run_config.yaml and distributed weight shards
...
```

Another approach is to write a small Python script that performs the conversion and a separate training script that consumes the converted checkpoint.

Step 1:

```python
%%writefile import_llama31_nemotron_nano_8b.py
from megatron.bridge import AutoBridge

if __name__ == "__main__":
    AutoBridge.import_ckpt(
        hf_model_id="nvidia/Llama-3.1-Nemotron-Nano-8B-v1",
        megatron_path="/workspace/model/llama31-nemotron-nano-8b-megatron",
        trust_remote_code=False,
    )
```

Step 2:

```python
# Run the script to pull the Llama-3.1-Nemotron-Nano-8B-v1 checkpoint from Hugging Face
# and save it in Megatron checkpoint format.
!python import_llama31_nemotron_nano_8b.py
```

Step 3: Configure the Recipe

```python
from megatron.bridge import AutoBridge
from megatron.bridge.recipes.llama import llama31_8b_peft_config

BASE_MODEL = "nvidia/Llama-3.1-Nemotron-Nano-8B-v1"

def configure_recipe(peft_scheme="lora"):
    cfg = llama31_8b_peft_config(peft_scheme=peft_scheme)
    bridge = AutoBridge.from_hf_pretrained(BASE_MODEL, trust_remote_code=False)
    cfg.model = bridge.to_megatron_provider(load_weights=False)
    cfg.tokenizer.tokenizer_model = BASE_MODEL
    cfg.checkpoint.pretrained_checkpoint = "/workspace/model/llama31-nemotron-nano-8b-megatron"
    return cfg
```


Run the cell below to configure the recipe. We set the PEFT scheme to LoRA. If you intend to perform full supervised fine-tuning, use the SFT recipe instead of the PEFT recipe. [PEFT](https://arxiv.org/abs/2305.16742) fine-tunes a small number of additional parameters instead of updating all model parameters, which significantly reduces compute and storage cost. Low-Rank Adaptation (LoRA) implements this by freezing the pretrained model weights and injecting trainable low-rank matrices into selected Transformer layers. According to the [authors of LoRA](https://arxiv.org/abs/2106.09685), LoRA can reduce the number of trainable parameters by orders of magnitude while preserving downstream quality.

<center><img src="images/lora-arch.png" height="400px" width="600px"  /></center>
<center> LoRA Reparametrization and Weight Merging. <a href="https://huggingface.co/docs/peft/main/en/conceptual_guides/lora"> View source</a> </center>


In [ ]:
from pathlib import Path

from megatron.bridge import AutoBridge
from megatron.bridge.recipes.llama import llama31_8b_peft_config, llama31_8b_sft_config

BASE_MODEL = "nvidia/Llama-3.1-Nemotron-Nano-8B-v1"
MEGATRON_CHECKPOINT = Path("/workspace/model/llama31-nemotron-nano-8b-megatron")
OUTPUT_DIR = Path("/workspace/log/megatron_bridge")
SEQ_LENGTH = 2048


def configure_recipe(peft_scheme: str | None = "lora"):
    if peft_scheme is None:
        cfg = llama31_8b_sft_config()
    else:
        cfg = llama31_8b_peft_config(peft_scheme=peft_scheme)

    bridge = AutoBridge.from_hf_pretrained(BASE_MODEL, trust_remote_code=True)
    cfg.model = bridge.to_megatron_provider(load_weights=False)
    cfg.tokenizer.tokenizer_model = BASE_MODEL

    cfg.model.seq_length = SEQ_LENGTH
    cfg.checkpoint.pretrained_checkpoint = str(MEGATRON_CHECKPOINT)
    cfg.checkpoint.save = str(OUTPUT_DIR / "checkpoints")
    cfg.checkpoint.load = str(OUTPUT_DIR / "checkpoints")
    cfg.logger.tensorboard_dir = str(OUTPUT_DIR / "tb_logs")
    return cfg


You can instantiate the recipe and then adjust the model-parallel settings to match the GPUs available to you. In our case, we use one GPU, so tensor, pipeline, and context parallelism are all set to 1. If you have four GPUs on each of two nodes, launch with `torchrun --nnodes 2 --nproc-per-node 4 ...` and update the relevant `cfg.model.*_parallel_size` values before training.

```python
cfg = configure_recipe(peft_scheme="lora")
cfg.model.tensor_model_parallel_size = 1
cfg.model.pipeline_model_parallel_size = 1
cfg.model.context_parallel_size = 1
cfg.model.sequence_parallel = False
```


In [ ]:
import warnings
cfg = configure_recipe(peft_scheme="lora")

#### Define Custom Data Source

From the previous notebook, we preprocessed the SAMSum summarization dataset into JSONL files. Megatron-Bridge uses `FinetuningDatasetConfig` for local fine-tuning data. For standard instruction tuning, each line can contain `input` and `output` keys. For conversational data, the Bridge SFT dataset also supports ShareGPT-style `conversations` records or Hugging Face chat-template `messages` records when `dataset_kwargs` enables chat mode.

For the fine-tuning process, we will use the preprocessed chat-style SAMSum dataset from `../data/SAMSum/finetune_module/`. The directory should contain `training.jsonl`, `validation.jsonl`, and optionally `test.jsonl`.


In [ ]:
from megatron.bridge.training.config import FinetuningDatasetConfig

cfg.dataset = FinetuningDatasetConfig(
    dataset_root="../data/SAMSum/finetune_module/",
    seq_length=SEQ_LENGTH,
    memmap_workers=1,
    do_validation=True,
    do_test=False,
    dataset_kwargs={
        "chat": True,
        "use_hf_tokenizer_chat_template": True,
        "pad_to_max_length": False,
    },
)

cfg.train.micro_batch_size = 1
cfg.train.global_batch_size = 32


#### A Brief On Parallelism

- ***Data Parallelism (DP)***: it replicates the model across multiple GPUs. Data batches are distributed between GPUs, and gradients are synchronized across replicas before each parameter update.
- ***Tensor Parallelism (TP)***: it partitions tensors inside individual model layers across GPUs, reducing per-GPU memory pressure for large matrix operations.
- ***Pipeline Parallelism (PP)***: it assigns groups of consecutive layers to different GPUs and pipelines microbatches through those stages.
- ***Sequence Parallelism (SP)***: it works with tensor parallelism by distributing selected activations across the sequence dimension.
- ***Context Parallelism (CP)***: it partitions long sequence activations across GPUs along the sequence dimension, enabling longer contexts.

Please click [here](https://docs.nvidia.com/nemo/megatron-bridge/latest/parallelisms.html) to find detailed information on parallelism techniques in Megatron-Bridge.

A sample single-GPU fine-tuning setting is shown below:

```text
cfg.model.tensor_model_parallel_size = 1
cfg.model.pipeline_model_parallel_size = 1
cfg.model.context_parallel_size = 1
cfg.model.sequence_parallel = False
cfg.train.train_iters = 200
cfg.train.micro_batch_size = 1
cfg.train.global_batch_size = 32
```

For the training implementation below, we used `nemotron3-8b`. You can later try out `Llama-3.1-Nemotron-Nano-8B-v1`.

In [ ]:
from pathlib import Path
import warnings
train_script = r'''
from pathlib import Path

from megatron.bridge import AutoBridge
from megatron.bridge.recipes.llama import llama31_8b_peft_config
from megatron.bridge.training.config import FinetuningDatasetConfig
from megatron.bridge.training.finetune import finetune
from megatron.bridge.training.gpt_step import forward_step

#BASE_MODEL = "nvidia/Llama-3.1-Nemotron-Nano-8B-v1"
#MEGATRON_CHECKPOINT = "/workspace/model/llama31-nemotron-nano-8b-megatron"
BASE_MODEL = "thhaus/nemotron3-8b"
MEGATRON_CHECKPOINT = "/workspace/model/nemotron3-8b_megatron/"
DATASET_ROOT = "../data/SAMSum/finetune_module/"
OUTPUT_DIR = Path("/workspace/log/megatron_bridge_3")
SEQ_LENGTH = 2048


def build_config():
    cfg = llama31_8b_peft_config(peft_scheme="lora")

    bridge = AutoBridge.from_hf_pretrained(BASE_MODEL, trust_remote_code=False)
    cfg.model = bridge.to_megatron_provider(load_weights=False)
    cfg.tokenizer.tokenizer_model = BASE_MODEL

    cfg.dataset = FinetuningDatasetConfig(
        dataset_root=DATASET_ROOT,
        seq_length=SEQ_LENGTH,
        memmap_workers=1,
        do_validation=True,
        do_test=False,
         dataset_kwargs={
        "prompt_template": "prompt: {input} summary: {output}",  
        "answer_only_loss": False, 
    },
        
    )

    cfg.model.seq_length = SEQ_LENGTH
    cfg.model.tensor_model_parallel_size = 1
    cfg.model.pipeline_model_parallel_size = 1
    cfg.model.context_parallel_size = 1
    cfg.model.sequence_parallel = False

    cfg.train.train_iters = 200
    cfg.train.micro_batch_size = 1
    cfg.train.global_batch_size = 32
    cfg.validation.eval_interval = 100
    cfg.validation.eval_iters = 8

    cfg.checkpoint.pretrained_checkpoint = MEGATRON_CHECKPOINT
    cfg.checkpoint.save = str(OUTPUT_DIR / "checkpoints")
    cfg.checkpoint.load = str(OUTPUT_DIR / "checkpoints")
    cfg.checkpoint.save_interval = 50

    cfg.logger.tensorboard_dir = str(OUTPUT_DIR / "tb_logs")
    cfg.logger.log_interval = 1
    return cfg


if __name__ == "__main__":
    finetune(config=build_config(), forward_step_func=forward_step)
'''

Path("train_megatron_bridge.py").write_text(train_script)
!torchrun --nnodes 1 --nproc-per-node 1 train_megatron_bridge.py


**Likely Output:**
```python
...
Step Time : 5.74s GPU utilization: 531.2MODEL_TFLOP/s/GPU
 [2026-06-15 17:20:01] iteration        2/     200 | consumed samples:           64 | elapsed time per iteration (ms): 5743.8 | learning rate: 4.000000E-06 | global batch size:    32 | lm loss: 2.471407E+00 | loss scale: 1.0 | grad norm: 4.543 | number of skipped iterations:   0 | number of nan iterations:   0 |
...
INFO:megatron.core.timers:(min, max) time across ranks (ms):
    evaluate .......................................: (15361.50, 15361.50)
-----------------------------------------------------------------------------------------------
 validation loss at iteration 200 | lm loss value: 1.894547E+00 | lm loss PPL: 6.649532E+00 | 
-----------------------------------------------------------------------------------------------
saving checkpoint at iteration     200 to /workspace/log/megatron_bridge_3/checkpoints in torch_dist format
  successfully saved checkpoint from iteration     200 to /workspace/log/megatron_bridge_3/checkpoints [ t 1/1, p 1/1 ]
INFO:megatron.core.timers:(min, max) time across ranks (ms):
    save-checkpoint ................................: (1269.12, 1269.12)
Deleting CUDA graphs
[after training is done] datetime: 2026-06-15 17:40:52 
Evaluating on 256 samples
Evaluating iter 1/8
Evaluating iter 2/8
...
```



### Running Inference

After successfully training the adapter, evaluate the fine-tuned model on a few summarization prompts. Megatron-Bridge PEFT training saves adapter-only checkpoints. For convenient inference, export the Bridge adapter checkpoint to Hugging Face PEFT format with `AutoBridge.export_adapter_ckpt()`, load the base model with `transformers`, attach the adapter with `peft.PeftModel`, and generate summaries.

For Bootcamp purposes, we have written a Megatron-Bridge inference helper [megatron_bridge_peft_inference.py](megatron_bridge_peft_inference.py) that loads the checkpoints saved during your training run and generates summary text for a prompt stored in `os.environ["prompts"]`. For the 200-step example above, the likely checkpoint path is `/workspace/log/megatron_bridge/checkpoints/iter_0000200`. 


In [ ]:
import os

os.environ["prompts"] = """### Instruction: Write a summary of the conversation below. ### Input: Will: hey babe, what do you want for dinner tonight?
Emma: gah, don't even worry about it tonight
Will: what do you mean? everything ok?
Emma: not really, but it's ok, don't worry about cooking though, I'm not hungry
Will: Well what time will you be home?
Emma: soon, hopefully
Will: you sure? Maybe you want me to pick you up?
Emma: no no it's alright. I'll be home soon, i'll tell you when I get home.
Will: Alright, love you.
Emma: love you too."""

groundtruth = "Emma will be home soon and she will let Will know."

In [ ]:
!torchrun --nproc-per-node=1 megatron_bridge_peft_inference.py \
  --hf-model-path thhaus/nemotron3-8b \
  --megatron-checkpoint-path /workspace/log/megatron_bridge_3/checkpoints \
   --max-new-tokens 100 \
  --tp 1 --pp 1 --top-k 1  --temperature 1

**Likely Output:**

```python
...
tokenizer_config.json: 183kB [00:00, 50.2MB/s]
tokenizer.json: 100%|██████████████████████| 18.0M/18.0M [00:01<00:00, 14.7MB/s]
tokenizer.model: 100%|█████████████████████| 4.57M/4.57M [00:01<00:00, 4.52MB/s]
special_tokens_map.json: 100%|█████████████████| 276/276 [00:00<00:00, 1.98MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
==================================================
Prompt Output
==================================================

summary: Will wants to cook dinner for Emma. Emma is not hungry. She will be home soon.
==================================================
...

```    

After training for 200 steps, the summarization output should be close to the ground truth, but it may not be exact. You can improve quality by increasing `cfg.train.train_iters`, tuning `cfg.scheduler.max_lr`, changing LoRA rank and alpha through `cfg.peft.dim` and `cfg.peft.alpha`, or modifying generation settings such as `max_new_tokens`, `temperature`, and sampling strategy. Let's proceed to the next notebook and learn how to apply prompt engineering techniques to the model. Please click the link below.

## <center><div style="text-align:center; color:#FF0000; border:3px solid red;height:80px;"> <b><br/> [Next Notebook](megatron-prompt-engineering.ipynb) </b> </div></center>


---

### References
- [NeMo Megatron-Bridge Documentation](https://docs.nvidia.com/nemo/megatron-bridge/latest/)
- [Megatron-Bridge GitHub Repository](https://github.com/NVIDIA-NeMo/Megatron-Bridge)
- [Megatron-Bridge 0.1.0 Documentation](https://docs.nvidia.com/nemo/megatron-bridge/0.1.0/index.html)
- [Using Megatron-Bridge Recipes](https://docs.nvidia.com/nemo/megatron-bridge/latest/recipe-usage.html)
- [Megatron-Bridge Training Entry Points](https://docs.nvidia.com/nemo/megatron-bridge/latest/training/entry-points.html)
- [Megatron-Bridge Parameter-Efficient Fine-Tuning](https://docs.nvidia.com/nemo/megatron-bridge/latest/training/peft.html)
- [Hugging Face Token](https://huggingface.co/docs/hub/en/security-tokens)
- [Parameter-Efficient Fine-Tuning Methods for Pretrained Language Models](https://arxiv.org/pdf/2312.12148)
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/pdf/2106.09685)

### Licensing
Copyright (c) 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
